In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [3]:
import warnings

import json
from numpy import random
from model.optimization import create_study, ObjectiveSuggestion, SuggestionValueType, eval_model
from model.dataset import get_dataframe

import lightgbm as lgb

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
warnings.filterwarnings("ignore")

In [ ]:
def run_tests(category: str, method: str, metric: str):
  train_data = get_dataframe(category=category, dataset="train")
  pvalues = json.loads(open(f"../../preprocessed/{category}/important_genes_{method}_{metric}.json").readline())
  chosen_genes = list(set([y["gene"] for x in [sex_values[:15] for subtype_items in pvalues.values() for sex_values in subtype_items.values()] for y in x]))
  print(f"Total chosen genes: {len(chosen_genes)}")

  _, model = create_study(
    name=f"lgbm_feature_selection_{method}_{metric}",
    model_factory=lambda **kwargs: lgb.LGBMClassifier(**kwargs, objective="multiclass", class_weight="balanced", verbosity=-1, n_jobs=-1),
    custom_dataset=(train_data[chosen_genes], train_data["subtype"]),
    suggestions=[
      ObjectiveSuggestion(
        value_type=SuggestionValueType.INT,
        param="n_estimators",
        param_range=(10, 4_000)
      ),
      ObjectiveSuggestion(
        value_type=SuggestionValueType.FLOAT,
        param="reg_alpha",
        param_range=(1e-5, 1),
        is_log=True
      ),
      ObjectiveSuggestion(
        value_type=SuggestionValueType.FLOAT,
        param="learning_rate",
        param_range=(1e-3, 3),
        is_log=True
      )
    ],
    report_test_results=False,
    trials=100
  )

  test_data = get_dataframe(category=category, dataset="test")
  eval_model(model, dataset=(test_data[chosen_genes], test_data["subtype"]), report=True)

In [5]:
run_tests(category="min_tpm_5", method="random_forest", metric="recall")

[I 2025-08-03 21:31:37,104] A new study created in Journal with name: lgbm_feature_selection_random_forest_recall


Total chosen genes: 254


[I 2025-08-03 21:32:17,088] Trial 2 finished with value: 0.8782332518778995 and parameters: {'n_estimators': 2494, 'reg_alpha': 0.008952387146016805, 'learning_rate': 0.21411431496759525}. Best is trial 2 with value: 0.8782332518778995.
[I 2025-08-03 21:34:52,517] Trial 4 finished with value: 0.8780523112522264 and parameters: {'n_estimators': 2177, 'reg_alpha': 0.00013065006527913288, 'learning_rate': 0.0014440532890880855}. Best is trial 2 with value: 0.8782332518778995.
[I 2025-08-03 21:35:45,733] Trial 3 finished with value: 0.8685136566341554 and parameters: {'n_estimators': 166, 'reg_alpha': 0.00011177774972795397, 'learning_rate': 0.1058393355578643}. Best is trial 2 with value: 0.8782332518778995.
[I 2025-08-03 21:35:46,396] Trial 8 finished with value: 0.883844117363928 and parameters: {'n_estimators': 2273, 'reg_alpha': 8.722626557928866e-05, 'learning_rate': 0.001102690628695906}. Best is trial 8 with value: 0.883844117363928.
[I 2025-08-03 21:35:51,807] Trial 6 finished wit

   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.854933  0.823591      0.832354           0.832354  0.854651


In [6]:
run_tests(category="min_tpm_5", method="random_forest", metric="f1")

[I 2025-08-03 21:59:19,562] A new study created in Journal with name: lgbm_feature_selection_random_forest_f1


Total chosen genes: 252


[I 2025-08-03 21:59:28,623] Trial 2 finished with value: 0.881550204471791 and parameters: {'n_estimators': 361, 'reg_alpha': 0.002629071661644785, 'learning_rate': 0.13794626252538247}. Best is trial 2 with value: 0.881550204471791.
[I 2025-08-03 21:59:44,437] Trial 1 finished with value: 0.39984516401551773 and parameters: {'n_estimators': 1858, 'reg_alpha': 0.7379172279052891, 'learning_rate': 1.3076313868264775}. Best is trial 2 with value: 0.881550204471791.
[I 2025-08-03 22:00:33,991] Trial 4 finished with value: 0.859552337337558 and parameters: {'n_estimators': 915, 'reg_alpha': 0.026164771814647877, 'learning_rate': 0.003808719435545283}. Best is trial 2 with value: 0.881550204471791.
[I 2025-08-03 22:00:48,880] Trial 7 finished with value: 0.8880597147090894 and parameters: {'n_estimators': 3276, 'reg_alpha': 4.452999455442642e-05, 'learning_rate': 0.029578202240246345}. Best is trial 7 with value: 0.8880597147090894.
[I 2025-08-03 22:02:05,628] Trial 5 finished with value: 0

   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.853826  0.820311      0.817879           0.817879  0.854651


In [7]:
run_tests(category="min_tpm_5", method="logistic", metric="recall")

[I 2025-08-03 22:39:31,951] A new study created in Journal with name: lgbm_feature_selection_logistic_recall


Total chosen genes: 239


[I 2025-08-03 22:39:44,472] Trial 1 finished with value: 0.8692833003643358 and parameters: {'n_estimators': 367, 'reg_alpha': 0.0003298269194076441, 'learning_rate': 0.10015194393298234}. Best is trial 1 with value: 0.8692833003643358.
[I 2025-08-03 22:40:01,851] Trial 4 finished with value: 0.8747611989571344 and parameters: {'n_estimators': 987, 'reg_alpha': 0.15258538533169394, 'learning_rate': 0.06765377579900773}. Best is trial 4 with value: 0.8747611989571344.
[I 2025-08-03 22:40:10,984] Trial 3 finished with value: 0.8722878768269575 and parameters: {'n_estimators': 2411, 'reg_alpha': 0.056832878608604635, 'learning_rate': 0.04811372757633227}. Best is trial 4 with value: 0.8747611989571344.
[I 2025-08-03 22:40:15,654] Trial 5 finished with value: 0.23589181965739886 and parameters: {'n_estimators': 1812, 'reg_alpha': 0.7234827993117717, 'learning_rate': 1.5029574883322638}. Best is trial 4 with value: 0.8747611989571344.
[I 2025-08-03 22:40:26,103] Trial 2 finished with value:

   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.860537   0.83916      0.836719           0.836719  0.860465


In [8]:
run_tests(category="min_tpm_5", method="logistic", metric="f1")

[I 2025-08-03 23:00:17,692] A new study created in Journal with name: lgbm_feature_selection_logistic_f1


Total chosen genes: 237


[I 2025-08-03 23:00:50,734] Trial 1 finished with value: 0.8632738883120734 and parameters: {'n_estimators': 2303, 'reg_alpha': 0.005659608866132291, 'learning_rate': 0.37585540566826603}. Best is trial 1 with value: 0.8632738883120734.
[I 2025-08-03 23:01:03,643] Trial 0 finished with value: 0.8710199778536041 and parameters: {'n_estimators': 2481, 'reg_alpha': 0.00030283628511487904, 'learning_rate': 0.11948872843771581}. Best is trial 0 with value: 0.8710199778536041.
[I 2025-08-03 23:01:06,758] Trial 3 finished with value: 0.8775357393814518 and parameters: {'n_estimators': 1097, 'reg_alpha': 0.006135778946921372, 'learning_rate': 0.06100156618159556}. Best is trial 3 with value: 0.8775357393814518.
[I 2025-08-03 23:01:16,234] Trial 8 finished with value: 0.8708814124738617 and parameters: {'n_estimators': 246, 'reg_alpha': 7.155471564479847e-05, 'learning_rate': 0.06331621531111212}. Best is trial 3 with value: 0.8775357393814518.
[I 2025-08-03 23:01:24,421] Trial 7 finished with 

   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.841352   0.80865       0.79894            0.79894  0.843023


Exception ignored in: <function ResourceTracker.__del__ at 0x104b4c4a0>
Traceback (most recent call last):
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1071584a0>
Traceback (most recent call last):
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py",